# Stage 4 — Paper roles

Stable cores and boundary entropy come FIRST (they produce `coassign_share.npy`, which the role scripts read); then degree/PageRank/betweenness, cohort percentiles, the role plane, null-adjusted influence, the full-precision role product with the strict census, and the roles table.

In [ ]:
import os, pathlib, sys
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "scripts").is_dir() and (root / "data").is_dir(): break
    root = root.parent
else:
    raise SystemExit("repository root (containing scripts/ and data/) not found within 6 levels")
os.chdir(root); print("working directory:", os.getcwd())
assert "igraph" in {m.split("==")[0] for m in os.popen(f"{sys.executable} -m pip list --format=freeze 2>/dev/null").read().split()}, \
    f"kernel {sys.executable} lacks python-igraph: select the grb-venv kernel (see notebooks/README.md)"

In [ ]:
CORPUS = "data/raw/ads_corpus_v2_core_frozen.jsonl"  # local-only frozen corpus (ADS terms); see README
import pathlib
HAVE_CORPUS = pathlib.Path(CORPUS).exists()
print("frozen corpus present:", HAVE_CORPUS)
RUN_LONG = False

Stable cores (100 Leiden runs — long). Produces `coassign_share.npy`.

In [ ]:
if RUN_LONG and HAVE_CORPUS:
    %run scripts/boundary_entropy.py
else:
    import numpy as np; s = np.load("data/communities/coassign_share.npy"); print("saved coassign_share:", s.shape)

Roles on the citation map (betweenness is the slow part).

In [ ]:
if RUN_LONG and HAVE_CORPUS:
    %run scripts/paper_roles.py
    %run scripts/roles_extend.py
    %run scripts/pagerank_null.py
else:
    print("skipped (long); products: paper_roles.json, paper_roles_table.csv.gz, roles_extended.json, pagerank_null.json")

Full-precision z and P for every paper plus the strict Guimerà–Amaral census (fast; this is the source the roles table formats from).

In [ ]:
if HAVE_CORPUS:
    %run scripts/roles_fullprecision.py
else:
    import json; print(json.load(open("data/communities/role_census.json"))["displayed_stable"])

In [ ]:
if HAVE_CORPUS:
    %run scripts/concept_roles.py
%run scripts/make_roles_table.py